<a href="https://colab.research.google.com/github/vinhdo19111999-hash/Start/blob/main/law.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-docx pandas scikit-learn openpyxl
!pip install docx2txt
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.7 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

import docx2txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path_word = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/84_2015_QH13_281961.docx'

In [ ]:
from docx import Document

def extract_all_text_from_docx(file_path):
    try:
        # docx2txt tự động đọc toàn bộ paragraph và table theo đúng thứ tự hiển thị
        full_text = docx2txt.process(file_path)
        return full_text
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return None

# Đọc lại file của bạn
file_path_word = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/84_2015_QH13_281961.docx'
full_text = extract_all_text_from_docx(file_path_word)

if full_text:
    print(f"Đã đọc file thành công, độ dài văn bản: {len(full_text)} ký tự")
    print("--- Văn bản mẫu ---")
    print(full_text[:500])  # Sẽ in ra đầy đủ phần Quốc hội, Cộng hòa...

Đã đọc file thành công, độ dài văn bản: 134340 ký tự
--- Văn bản mẫu ---
QUỐC HỘI
-------

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc 
---------------

Luật số: 84/2015/QH13

Hà Nội, ngày 25 tháng 06 năm 2015

 

LUẬT

AN TOÀN, VỆ SINH LAO ĐỘNG

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật an toàn, vệ sinh lao động.

Chương I

QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh

Luật này quy định việc bảo đảm an toàn, vệ sinh lao động; chính sách, chế độ đối với người bị tai nạn lao động, bệnh nghề nghiệp; trách nhiệm v


In [ ]:
#Sử dụng regex để chuẩn hóa các ký tự, xử lý các lỗi font chữ thường gặp...

def standardize_text(text): #Chuẩn hóa văn bản, xử lý các lỗi font và ký tự đặc biệt
    if not text: #Kiểm tra nếu đầu vào là chuỗi rỗng, None hoặc False
        return "" #Trả về chuỗi rỗng ngay lập tức để tránh lỗi khi xử lý

    text = re.sub(r'[\x82\x84\x85\x91\x92\x93\x94\x96\x97]', ' ', text) #Thay thế các ký tự bị lỗi font thường gặp thành dấu cách

    text = re.sub(r'[ \t]+', ' ', text).strip()#Thay thế nhiều khoảng trắng/tab liền nhau thành 1 khoảng trắng, và xóa khoảng trắng ở 2 đầu chuỗi

    text = re.sub(r'[^\w\s\.\,\;\n]', ' ', text)#(Tùy chọn) Bỏ comment dòng này nếu muốn loại bỏ sạch các ký tự đặc biệt, chỉ giữ lại chữ, số, khoảng trắng và các dấu . , ;

    return text #Trả về kết quả chuỗi văn bản đã được chuẩn hóa hoàn tất

# Áp dụng chuẩn hóa cho văn bản đã đọc
if full_text:#Kiểm tra xem biến full_text (văn bản đã đọc) có chứa dữ liệu hay không
    full_text_standardized = standardize_text(full_text) #Gọi hàm chuẩn hóa và lưu kết quả vào biến full_text_standardized
    print(f"Văn bản đã được chuẩn hóa, độ dài: {len(full_text_standardized)} ký tự")#In ra màn hình xác nhận kèm tổng số lượng ký tự của văn bản mới
    print(full_text_standardized[:500])

Văn bản đã được chuẩn hóa, độ dài: 134340 ký tự
QUỐC HỘI
       

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập   Tự do   Hạnh phúc 
               

Luật số  84 2015 QH13

Hà Nội, ngày 25 tháng 06 năm 2015

 

LUẬT

AN TOÀN, VỆ SINH LAO ĐỘNG

Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;

Quốc hội ban hành Luật an toàn, vệ sinh lao động.

Chương I

QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh

Luật này quy định việc bảo đảm an toàn, vệ sinh lao động; chính sách, chế độ đối với người bị tai nạn lao động, bệnh nghề nghiệp; trách nhiệm v


In [ ]:
# Ô này thay thế hoàn toàn ô code cũ (parse_law_text_to_chunks bị lỗi) - giữ nguyên 6 ô code phía trên
# 2 lỗi gốc đã sửa: (1) mẫu Chương yêu cầu dấu "." sai vì "Chương I" và tên nằm 2 dòng riêng
#                    (2) mẫu Điểm tìm dấu ")" nhưng standardize_text() đã xóa mất dấu ")" từ trước

def parse_law_text_to_chunks(text):  # ham chinh, nhan van ban da chuan hoa, tra ve DataFrame co cau truc
    danh_sach_doan = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]  # tach van ban thanh tung doan, dua vao dong trong (docx2txt phan cach doan bang \n\n)

    MAU_CHUONG = re.compile(r"^Chương\s+([IVXLCDM]+)$")                   # mau nhan dien dong "Chương I", "Chương II"... KHONG co dau cham hay ten tren cung dong
    MAU_DIEU = re.compile(r"^Điều\s+(\d+)\.\s*(.+)$")                     # mau nhan dien dong "Điều 6. Ten dieu", nhom 1 la so dieu, nhom 2 la ten
    MAU_KHOAN = re.compile(r"^(\d+)\.\s*(.+)$")                           # mau nhan dien mot Khoan dang "1. Noi dung", dau cham khong bi standardize_text xoa
    MAU_DIEM = re.compile(r"^([a-zđươ]{1,2})\s{2,}(.+)$", re.IGNORECASE)  # mau nhan dien Diem: sau khi mat dau ")" thi "a) Noi dung" thanh "a  Noi dung" (2 khoang trang)

    rows = []                       # list chua cac dict, moi dict la 1 dong ket qua cuoi cung (1 Khoan/Diem/cau mo dau)
    chuong_id = None                # so hieu Chuong dang xu ly (vd "I", "II"), cap nhat khi gap tieu de Chuong moi
    chuong_ten = None               # ten day du cua Chuong dang xu ly (vd "QUY ĐỊNH CHUNG")
    cho_ten_chuong = False          # co bao hieu doan tiep theo chinh la TEN cua Chuong vua gap
    dieu_id = None                  # so hieu Dieu dang xu ly (vd "6"), cap nhat khi gap tieu de Dieu moi
    dieu_ten = None                 # ten day du cua Dieu dang xu ly
    khoan_id = None                 # so hieu Khoan dang xu ly, dung de gan Diem (a, b, c) vao dung Khoan cha
    intro_buf = []                  # list tam gom cac cau mo dau cua 1 Dieu (truoc khi gap Khoan dau tien)

    def citation(khoan=None, diem=None):                  # ham con tao chuoi trich dan day du dua tren Dieu/Khoan/Diem hien tai
        s = f"Điều {dieu_id}"                               # phan bat buoc: luon co so Dieu
        if khoan is not None:                               # neu co Khoan thi them vao
            s += f", Khoản {khoan}"                          # noi them "Khoan X"
        if diem is not None:                                # neu co Diem thi them vao
            s += f", Điểm {diem}"                            # noi them "Diem x"
        return f"{s} - Luật An toàn, vệ sinh lao động số 84/2015/QH13"  # ghep ten day du bo luat vao cuoi

    def flush_intro():                                     # ham con day cau mo dau da gom duoc thanh 1 dong "dieu_intro"
        if intro_buf:                                        # chi thuc hien neu thuc su co noi dung can ghi
            rows.append({                                     # them 1 dong moi vao ket qua
                'chuong_id': chuong_id, 'chuong_ten': chuong_ten,      # gan Chuong hien tai
                'dieu_id': dieu_id, 'dieu_ten': dieu_ten,              # gan Dieu hien tai
                'khoan_id': None, 'diem_id': None,                     # dong intro khong thuoc Khoan/Diem nao
                'cap_do': 'dieu_intro',                                # danh dau cap do
                'noi_dung': ' '.join(intro_buf).strip(),               # noi cac cau mo dau thanh 1 doan hoan chinh
                'full_citation': citation(),                           # trich dan chi voi so Dieu
            })
            intro_buf.clear()                                # don sach list tam cho Dieu tiep theo

    for doan in danh_sach_doan:                            # duyet qua tung doan theo dung thu tu trong file

        m = MAU_CHUONG.match(doan)                          # kiem tra doan co phai tieu de Chuong khong
        if m:                                               # neu dung la tieu de Chuong moi
            flush_intro()                                    # ghi not cau mo dau cua Dieu truoc do (neu con)
            chuong_id = m.group(1)                           # cap nhat so hieu Chuong
            cho_ten_chuong = True                            # bat co cho ten Chuong o dong ke tiep
            dieu_id = None                                   # reset Dieu vi sang Chuong moi
            khoan_id = None                                  # reset Khoan vi sang Chuong moi
            continue                                         # sang doan tiep theo

        if cho_ten_chuong:                                  # neu dong truoc la tieu de Chuong, dong nay la ten Chuong
            chuong_ten = doan.strip()                        # luu ten Chuong
            cho_ten_chuong = False                           # tat co
            continue                                         # sang doan tiep theo

        m = MAU_DIEU.match(doan)                            # kiem tra doan co phai tieu de Dieu khong
        if m:                                               # neu dung la tieu de Dieu moi
            flush_intro()                                    # ghi not cau mo dau cua Dieu TRUOC do (neu con)
            dieu_id = m.group(1)                             # cap nhat so hieu Dieu
            dieu_ten = m.group(2).strip()                    # cap nhat ten Dieu
            khoan_id = None                                  # reset Khoan vi sang Dieu moi
            continue                                         # sang doan tiep theo

        if dieu_id is None:                                 # neu chua tung gap Dieu nao (dang o phan mo dau van ban)
            continue                                         # bo qua hoan toan doan nay

        m_diem = MAU_DIEM.match(doan)                       # kiem tra doan co phai mot Diem khong
        m_khoan = MAU_KHOAN.match(doan)                     # kiem tra doan co phai mot Khoan khong

        if (m_diem or m_khoan) and intro_buf:               # neu sap ghi Khoan/Diem DAU TIEN ma con cau mo dau chua ghi
            flush_intro()                                    # ghi not cau mo dau NGAY LUC NAY, dat truoc Khoan 1

        if m_diem and khoan_id is not None:                 # neu la Diem VA da co Khoan cha dang xu ly
            rows.append({                                     # them dong moi dai dien cho Diem nay
                'chuong_id': chuong_id, 'chuong_ten': chuong_ten,
                'dieu_id': dieu_id, 'dieu_ten': dieu_ten,
                'khoan_id': khoan_id,                          # gan Khoan cha
                'diem_id': m_diem.group(1).strip(),            # ky hieu Diem (a, b, c...)
                'cap_do': 'diem',                              # danh dau cap do Diem
                'noi_dung': m_diem.group(2).strip(),           # noi dung cua Diem
                'full_citation': citation(khoan_id, m_diem.group(1).strip()),  # trich dan day du Dieu-Khoan-Diem
            })
            continue                                         # sang doan tiep theo

        if m_khoan:                                         # neu la mot dong Khoan moi
            khoan_id = m_khoan.group(1)                      # cap nhat so Khoan hien tai
            rows.append({                                     # them dong moi dai dien cho Khoan nay
                'chuong_id': chuong_id, 'chuong_ten': chuong_ten,
                'dieu_id': dieu_id, 'dieu_ten': dieu_ten,
                'khoan_id': khoan_id, 'diem_id': None,         # Khoan khong co Diem rieng
                'cap_do': 'khoan',                             # danh dau cap do Khoan
                'noi_dung': m_khoan.group(2).strip(),          # noi dung cua Khoan
                'full_citation': citation(khoan_id),           # trich dan Dieu-Khoan
            })
            continue                                         # sang doan tiep theo

        if khoan_id is None:                                # neu doan nay khong khop Khoan/Diem VA chua co Khoan nao
            intro_buf.append(doan.strip())                   # day la cau mo dau cua Dieu, gom tam lai
        else:                                                # neu da co Khoan truoc do roi
            rows.append({                                     # day la cau bo sung/ket luan cua Dieu, van ghi thanh dong rieng
                'chuong_id': chuong_id, 'chuong_ten': chuong_ten,
                'dieu_id': dieu_id, 'dieu_ten': dieu_ten,
                'khoan_id': khoan_id, 'diem_id': None,
                'cap_do': 'khoan_bo_sung',                     # danh dau day la cau bo sung
                'noi_dung': doan.strip(),                      # noi dung nguyen van cua cau bo sung
                'full_citation': citation(),                   # trich dan chi voi so Dieu
            })

    flush_intro()                                           # sau vong lap, ghi not cau mo dau con lai cua Dieu CUOI CUNG
    df = pd.DataFrame(rows)                                 # chuyen toan bo list dict thanh DataFrame
    df.insert(0, 'id', range(1, len(df) + 1))               # them cot id danh so thu tu tang dan tu 1

    df['ten_van_ban'] = 'Luật An toàn, vệ sinh lao động 2015'     # cot metadata: ten van ban, giong nhau moi dong
    df['ngay_ban_hanh'] = 'ngày 25 tháng 06 năm 2015'             # cot metadata: ngay ban hanh
    df['co_quan_ban_hanh'] = 'Quốc Hội'                            # cot metadata: co quan ban hanh
    df['so_hieu'] = 'Luật số: 84/2015/QH13'                        # cot metadata: so hieu van ban
    return df                                               # tra ve DataFrame hoan chinh

# Chay pipeline va luu file - giu dung duong dan Drive nhu ban da dung
if full_text_standardized:                                  # kiem tra van ban da chuan hoa co ton tai khong
    df_chunks = parse_law_text_to_chunks(full_text_standardized)  # goi ham chinh de phan tich cau truc luat
    print(f"Đã tạo {len(df_chunks)} chunks.")                # in tong so dong de kiem tra nhanh
    print(df_chunks['cap_do'].value_counts())                # in phan bo so luong theo tung cap_do

    file_path_csv = '/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/law_dataset_chunks.csv'  # duong dan luu file, sua lai neu can
    df_chunks.to_csv(file_path_csv, index=False, encoding='utf-8-sig')  # xuat CSV bang dau phay chuan (KHONG dung sep='|' nhu truoc)
    print(f"Đã lưu file tại: {file_path_csv}")               # thong bao hoan tat
else:
    print("Không thể parse văn bản do lỗi đọc file.")        # thong bao neu van ban dau vao bi rong

    print(df_chunks['cap_do'].value_counts())   # phải có đủ: khoan 358, diem 201, khoan_bo_sung 36, dieu_intro 11
    print(df_chunks['chuong_id'].isna().sum())  # phải bằng 0

Đã tạo 606 chunks.
cap_do
khoan            358
diem             201
khoan_bo_sung     36
dieu_intro        11
Name: count, dtype: int64
Đã lưu file tại: /content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/law_dataset_chunks.csv


In [ ]:
import re                                                   # thu vien regex, dung de bat so Dieu/Khoan/Diem trong cau hoi
import pandas as pd                                         # xu ly bang du lieu

df_chunks = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/law_dataset_chunks.csv')           # doc file du lieu luat da tao o cac buoc truoc

# ============================================================================
# BUOC 1 - NHAN DIEN SO DIEU/KHOAN/DIEM TRONG CAU HOI CUA NGUOI DUNG
# ============================================================================
def trich_xuat_dieu_khoan_diem(cau_hoi: str):               # ham nhan cau hoi tho, tra ve (so_dieu, so_khoan, ky_hieu_diem)
    m_dieu = re.search(r'[Đđ]i[eề]u\s*(?:số\s*)?(\d+)', cau_hoi)   # bat "Điều 6", "điều số 6", khong phan biet hoa/thuong
    m_khoan = re.search(r'[Kk]ho[ảa]n\s*(?:số\s*)?(\d+)', cau_hoi) # bat "Khoản 2", "khoan so 2"
    m_diem = re.search(r'[Đđ]i[ểe]m\s*([a-zđươA-ZĐƯƠ])\b', cau_hoi) # bat "Điểm a", "diem a"

    so_dieu = int(m_dieu.group(1)) if m_dieu else None       # chuyen ve so nguyen neu tim thay, khong thi None
    so_khoan = int(m_khoan.group(1)) if m_khoan else None     # tuong tu cho khoan
    ky_hieu_diem = m_diem.group(1).lower() if m_diem else None  # chuan hoa chu thuong cho diem (a, b, c...)

    return so_dieu, so_khoan, ky_hieu_diem                    # tra ve bo 3 gia tri, phan tu nao khong tim thay se la None


# ============================================================================
# BUOC 2 - TRA CUU TRUC TIEP TREN DU LIEU DUA VAO SO DIEU/KHOAN/DIEM DA BAT DUOC
# ============================================================================
def tra_cuu_truc_tiep(so_dieu: int, so_khoan: int = None, ky_hieu_diem: str = None):
    ket_qua = df_chunks[df_chunks['dieu_id'] == so_dieu]      # loc truoc theo so Dieu, day la dieu kien bat buoc

    if ket_qua.empty:                                         # neu khong tim thay Dieu nao trung so nay trong du lieu
        return None                                            # tra ve None, bao hieu khong tim thay - de ham goi biet ma fallback

    if so_khoan is not None:                                  # neu nguoi dung co nhac den so Khoan cu the
        ket_qua = ket_qua[(ket_qua['khoan_id'] == so_khoan) | (ket_qua['khoan_id'].isna() & (ket_qua['cap_do']=='dieu_intro'))]
        # dong tren: giu lai cac dong thuoc dung Khoan nay (ca khoan chinh lan cac diem con), BO QUA dieu_intro tru khi khong co khoan nao khac

    if ky_hieu_diem is not None:                              # neu nguoi dung co nhac den Diem cu the (vd "diem a")
        ket_qua = ket_qua[(ket_qua['diem_id'] == ky_hieu_diem) | (ket_qua['cap_do'] == 'khoan')]
        # giu lai dong Diem trung ky hieu, VA giu lai dong Khoan cha (de nguoi dung thay ca ngu canh cau dan "Khoan X quy dinh nhung diem sau:")

    return ket_qua.sort_values('id')                          # sap xep theo dung thu tu id goc, dam bao dieu_intro/khoan/diem hien dung trinh tu


# ============================================================================
# BUOC 3 - DINH DANG KET QUA THANH CAU TRA LOI DE DOC
# ============================================================================
def dinh_dang_ket_qua(ket_qua: pd.DataFrame) -> str:
    if ket_qua is None or ket_qua.empty:                      # neu khong co ket qua nao (Dieu/Khoan/Diem khong ton tai trong luat)
        return "Không tìm thấy quy định tương ứng trong Luật An toàn, vệ sinh lao động 2015."

    dong_van_ban = []                                          # list chua tung dong noi dung se ghep lai thanh cau tra loi
    for _, row in ket_qua.iterrows():                          # duyet qua tung dong ket qua da loc duoc, theo dung thu tu
        if row['cap_do'] == 'diem':                             # neu day la mot dong cap do Diem
            dong_van_ban.append(f"  {row['diem_id']}) {row['noi_dung']}")  # thut le va them ky hieu "a)" phia truoc
        elif row['cap_do'] == 'khoan':                          # neu day la mot dong cap do Khoan
            dong_van_ban.append(f"{int(row['khoan_id'])}. {row['noi_dung']}")  # them so khoan phia truoc, khong thut le
        else:                                                   # cac truong hop con lai (dieu_intro, khoan_bo_sung)
            dong_van_ban.append(row['noi_dung'])                # giu nguyen noi dung, khong them tien to

    citation = ket_qua.iloc[0]['full_citation'] if len(ket_qua) == 1 else f"Điều {int(ket_qua.iloc[0]['dieu_id'])} - {ket_qua.iloc[0]['ten_van_ban']} số {ket_qua.iloc[0]['so_hieu'].replace('Luật số: ','')}"
    # neu ket qua chi co 1 dong (da xac dinh ro Khoan/Diem) thi dung citation co san; neu tra ve nguyen ca Dieu thi tu tao citation cap Dieu

    tieu_de = f"📖 {citation}\n"                                # dong tieu de cua cau tra loi, kem bieu tuong cho de nhin
    noi_dung_day_du = "\n".join(dong_van_ban)                   # ghep tat ca cac dong noi dung lai, moi dong 1 dong rieng
    return tieu_de + noi_dung_day_du                            # tra ve chuoi hoan chinh: tieu de + noi dung


# ============================================================================
# BUOC 4 - HAM CHINH: NHAN CAU HOI, TRA VE CAU TRA LOI (HOAC BAO HIEU KHONG XU LY DUOC)
# ============================================================================
def bai_toan_1_tra_cuu_truc_tiep(cau_hoi: str):
    so_dieu, so_khoan, ky_hieu_diem = trich_xuat_dieu_khoan_diem(cau_hoi)  # buoc 1: bat so Dieu/Khoan/Diem tu cau hoi

    if so_dieu is None:                                        # neu cau hoi KHONG nhac den so Dieu nao ca
        return None                                             # tra ve None -> day la tin hieu de chuyen sang Bai toan 2 (tra cuu ngu nghia)

    ket_qua = tra_cuu_truc_tiep(so_dieu, so_khoan, ky_hieu_diem)  # buoc 2: loc du lieu dua tren so da bat duoc
    return dinh_dang_ket_qua(ket_qua)                           # buoc 3: dinh dang thanh cau tra loi de doc, roi tra ve


# ============================================================================
# THU NGHIEM VOI MOT SO CAU HOI MAU
# ============================================================================
cac_cau_hoi_thu = [
    "Điều 6 quy định gì?",
    "Điều 6 khoản 2 nói gì?",
    "Điều 2",
    "Điều 999 có nội dung gì?",
    "Bị tai nạn lao động thì được hưởng gì?",
]
for cau_hoi in cac_cau_hoi_thu:
    print("="*80)
    print("CÂU HỎI:", cau_hoi)
    print("-"*80)
    ket_qua = bai_toan_1_tra_cuu_truc_tiep(cau_hoi)
    print(ket_qua if ket_qua else "[KHÔNG PHÁT HIỆN SỐ ĐIỀU -> chuyển sang Bài toán 2]")


CÂU HỎI: Điều 6 quy định gì?
--------------------------------------------------------------------------------
📖 Điều 6 - Luật An toàn, vệ sinh lao động 2015 số 84/2015/QH13
1. Người lao động làm việc theo hợp đồng lao động có quyền sau đây
  a) Được bảo đảm các điều kiện làm việc công bằng, an toàn, vệ sinh lao động; yêu cầu người sử dụng lao động có trách nhiệm bảo đảm điều kiện làm việc an toàn, vệ sinh lao động trong quá trình lao động, tại nơi làm việc;
  b) Được cung cấp thông tin đầy đủ về các yếu tố nguy hiểm, yếu tố có hại tại nơi làm việc và những biện pháp phòng, chống; được đào tạo, huấn luyện về an toàn, vệ sinh lao động;
  c) Được thực hiện chế độ bảo hộ lao động, chăm sóc sức khỏe, khám phát hiện bệnh nghề nghiệp; được người sử dụng lao động đóng bảo hiểm tai nạn lao động, bệnh nghề nghiệp; được hưởng đầy đủ chế độ đối với người bị tai nạn lao động, bệnh nghề nghiệp; được trả phí khám giám định thương tật, bệnh tật do tai nạn lao động, bệnh nghề nghiệp; được chủ động đi k

In [ ]:
import re
import pandas as pd
from underthesea import word_tokenize                        # tach tu tieng Viet, giai quyet loi trung am tiet vo tinh
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df_chunks = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/law_retrieval/data/law_dataset_chunks.csv')

# ============================================================================
# BUOC 0 (MOI) - GHEP NGU CANH KHOAN CHA VAO CAC DONG DIEM/KHOAN_BO_SUNG QUA NGAN
# ============================================================================
def tao_van_ban_de_tim_kiem(df: pd.DataFrame) -> pd.Series:
    van_ban_tim_kiem = []                                       # list chua chuoi van ban DUNG DE VECTOR HOA, khac voi cot noi_dung goc dung de HIEN THI
    for idx, row in df.iterrows():                               # duyet qua tung dong trong du lieu
        if row['cap_do'] in ('diem', 'khoan_bo_sung') and pd.notna(row['khoan_id']):  # neu la dong Diem hoac cau bo sung, VA co Khoan cha
            dong_khoan_cha = df[(df['dieu_id'] == row['dieu_id']) &                    # tim dong Khoan cha tuong ung
                                 (df['khoan_id'] == row['khoan_id']) &
                                 (df['cap_do'] == 'khoan')]
            cau_dan = dong_khoan_cha.iloc[0]['noi_dung'] if not dong_khoan_cha.empty else ""  # lay noi dung cau dan cua Khoan cha (neu co)
            van_ban_tim_kiem.append(f"{cau_dan} {row['noi_dung']}")   # noi cau dan + noi dung that su, cho du ngu canh de vector hoa
        else:                                                     # cac truong hop con lai (dieu_intro, khoan) da du ngu canh, giu nguyen
            van_ban_tim_kiem.append(row['noi_dung'])
    return pd.Series(van_ban_tim_kiem, index=df.index)             # tra ve Series cung do dai, cung thu tu voi DataFrame goc

df_chunks['van_ban_tim_kiem'] = tao_van_ban_de_tim_kiem(df_chunks)  # them cot moi, CHI dung de tim kiem, khong thay the cot noi_dung hien thi


# ============================================================================
# BUOC 1 (SUA) - TACH TU TIENG VIET BANG UNDERTHESEA TRUOC KHI VECTOR HOA
# ============================================================================
def tach_tu(van_ban: str) -> str:
    return word_tokenize(str(van_ban).lower(), format="text")    # vd "thời tiết" -> "thời_tiết" (1 token), "chi tiết" -> "chi_tiết" (1 token khac)
    # nho gach noi "_", 2 cum nay KHONG CON trung nhau o muc am tiet nua - day chinh la diem sua loi cot loi

HU_TU = """và của cho theo là các những này đó khi nếu thì được có không phải để trong
với từ đến bị do vì như sau trước mà nào nên hay hoặc tại về sẽ đã một hai ba
việc theo quy_định tại khoản điều chi_tiết""".split()             # danh sach hu tu, viet theo dang co gach noi vi da qua tach tu

corpus_tho = df_chunks['van_ban_tim_kiem'].astype(str).tolist()   # lay cot van ban DA GHEP NGU CANH (khong phai noi_dung tho)
corpus_da_tach_tu = [tach_tu(v) for v in corpus_tho]               # tach tu tung dong, tra ve list chuoi da co gach noi giua cac tu ghep

tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words=HU_TU,
    token_pattern=r"(?u)\b\w+\b",
)
X_corpus = tfidf_vec.fit_transform(corpus_da_tach_tu)              # hoc tu vung va vector hoa TREN VAN BAN DA TACH TU (khac ban truoc)


# ============================================================================
# BUOC 2 - HAM TRA CUU (GIU NGUYEN LOGIC, CHI DOI DAU VAO DA QUA TACH TU)
# ============================================================================
def bai_toan_2_tra_cuu_ngu_nghia(cau_hoi: str, top_k: int = 3, nguong_diem_toi_thieu: float = 0.05):
    cau_hoi_da_tach_tu = tach_tu(cau_hoi)                          # QUAN TRONG: cau hoi cung phai duoc tach tu GIONG HET cach xu ly corpus
    vector_cau_hoi = tfidf_vec.transform([cau_hoi_da_tach_tu])
    diem_tuong_dong = cosine_similarity(vector_cau_hoi, X_corpus)[0]
    chi_so_top_k = diem_tuong_dong.argsort()[::-1][:top_k]

    ket_qua = []
    for idx in chi_so_top_k:
        diem = diem_tuong_dong[idx]
        if diem < nguong_diem_toi_thieu:
            continue
        dong = df_chunks.iloc[idx]
        ket_qua.append({
            'diem_tuong_dong': round(float(diem), 4),
            'full_citation': dong['full_citation'],
            'noi_dung': dong['noi_dung'],                          # HIEN THI van dung cot noi_dung GOC (khong dung ban da ghep ngu canh)
        })
    return ket_qua


def dinh_dang_ket_qua_bai_toan_2(ket_qua: list) -> str:
    if not ket_qua:
        return "Không tìm thấy quy định nào đủ liên quan trong Luật An toàn, vệ sinh lao động 2015."
    dong_hien_thi = []
    for i, r in enumerate(ket_qua, start=1):
        dong_hien_thi.append(f"{i}. 📖 {r['full_citation']} (độ liên quan: {r['diem_tuong_dong']})\n   {r['noi_dung']}")
    return "\n\n".join(dong_hien_thi)


# ============================================================================
# THU NGHIEM LAI DUNG 6 CAU HOI CU DE SO SANH TRUC TIEP
# ============================================================================
cac_cau_hoi_thu = [
    "Bị tai nạn lao động thì phải làm gì?",
    "Người sử dụng lao động có phải mua bảo hiểm tai nạn lao động không?",
    "Ai có trách nhiệm điều tra tai nạn lao động?",
    "Làm việc trong môi trường độc hại có được hưởng phụ cấp gì không?",
    "Người lao động có quyền từ chối làm việc nguy hiểm không?",
    "Thời tiết hôm nay thế nào?",
]
for cau_hoi in cac_cau_hoi_thu:
    print("="*90)
    print("CÂU HỎI:", cau_hoi)
    print("-"*90)
    ket_qua = bai_toan_2_tra_cuu_ngu_nghia(cau_hoi, top_k=3)
    print(dinh_dang_ket_qua_bai_toan_2(ket_qua))

# ============================================================================
# CAI TIEN THEM - NEU KET QUA TOP-1 LA DONG "KHOAN" (CAU DAN CHUNG CHUNG),
# TU DONG MO RONG HIEN THI CAC DIEM CON DE NGUOI DUNG THAY DAP AN CU THE
# ============================================================================
def mo_rong_neu_la_khoan_cha(ket_qua: list) -> list:
    ket_qua_moi = []                                            # list ket qua sau khi mo rong (neu can)
    for r in ket_qua:                                             # duyet qua tung ket qua tra ve tu Buoc 2
        dong_goc = df_chunks[df_chunks['full_citation'] == r['full_citation']]  # tim lai dong goc trong df_chunks
        if dong_goc.empty:
            ket_qua_moi.append(r); continue
        dong_goc = dong_goc.iloc[0]
        if dong_goc['cap_do'] == 'khoan':                          # neu day la mot dong Khoan (co the chi la cau dan chung chung)
            cac_diem_con = df_chunks[(df_chunks['dieu_id'] == dong_goc['dieu_id']) &   # tim tat ca Diem con cung Khoan nay
                                       (df_chunks['khoan_id'] == dong_goc['khoan_id']) &
                                       (df_chunks['cap_do'] == 'diem')]
            if not cac_diem_con.empty:                              # neu Khoan nay thuc su co Diem con (khong phai Khoan doc lap)
                noi_dung_mo_rong = r['noi_dung'] + "\n" + "\n".join(
                    f"   {d['diem_id']}) {d['noi_dung']}" for _, d in cac_diem_con.iterrows())  # ghep them tung Diem, thut le
                r = {**r, 'noi_dung': noi_dung_mo_rong}              # tao ban sao dict voi noi_dung da mo rong
        ket_qua_moi.append(r)
    return ket_qua_moi


def bai_toan_2_full(cau_hoi: str, top_k: int = 3):
    ket_qua = bai_toan_2_tra_cuu_ngu_nghia(cau_hoi, top_k=top_k)   # goi ham tra cuu nhu cu
    ket_qua = mo_rong_neu_la_khoan_cha(ket_qua)                     # ap dung buoc mo rong Khoan cha -> Diem con
    return dinh_dang_ket_qua_bai_toan_2(ket_qua)


print("\n\n" + "#"*90)
print("# TEST LAI SAU KHI THEM BUOC MO RONG KHOAN CHA")
print("#"*90)
print(bai_toan_2_full("Người lao động có quyền từ chối làm việc nguy hiểm không?", top_k=1))

NameError: name 'pd' is not defined